# RAG: Retrieval Augmented Generation

 Este notebook implementa un sistema RAG completo desde cero, sin frameworks de alto
 nivel que oculten los internals.

 **Stack:**
 - **Embeddings:** Google Gemini API *o* `sentence-transformers` (tú eliges)
 - **Vector store (internals):** FAISS manual
 - **Vector store (producción):** Qdrant 
 - **LLM de generación:** Google Gemini *o* Qwen2.5 (tú eliges)
 - **Evaluación:** DeepEval

| Parte | Tema |
|---|---|
| 0 | Setup y elección de backend de embeddings |
| 1 | El problema: LLM sin contexto |
| 2 | Internals: ¿qué es un embedding y un vector store? |
| 3 | Estrategias de chunking |
| 4 | RAG completo con Qdrant |
| 5 | Tuning: prompts y thresholds |
| 6 | Evaluación con DeepEval |

# Instalación de Ollama

In [ ]:
%%bash
curl -fsSL https://ollama.ai/install.sh | sh

In [ ]:
%%bash
ollama --version

### Descarga del Modelo TinyLlama
 
Descarga el modelo Qwen2.5-7b-Instruct 


In [ ]:
%%bash
ollama pull qwen2.5:7b-instruct

In [57]:
import os
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from dotenv import load_dotenv
import json
from pathlib import Path

In [2]:
load_dotenv()
 
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")
assert GOOGLE_API_KEY, "Falta GOOGLE_API_KEY en el .env"

print("✅ Variables de entorno cargadas")

✅ Variables de entorno cargadas


### 0.1 Elección del backend de embeddings
| Backend | Modelo | Dimensión | Requiere internet | Costo |
|---|---|---|---|---|
| `"gemini"` | `text-embedding-004` | 768 | ✅ | Free tier |
| `"local"` | `all-MiniLM-L6-v2` | 384 | Solo primera vez | Gratis |
Cambia `EMBEDDING_BACKEND` según tu preferencia.

In [11]:
# CONFIGURA AQUÍ
EMBEDDING_BACKEND = "local"   # "gemini" | "local"

In [ ]:
class EmbeddingService:
    """
    Wrapper unificado para obtener embeddings.
    Independientemente del backend, el resto del notebook siempre llama:
        service.embed(texts: List[str]) -> List[List[float]]
    """
 
    def __init__(self, backend: str = "gemini"):
        self.backend = backend
 
        if backend == "gemini":
            from google import genai
            self.client = genai.Client(api_key=GOOGLE_API_KEY)
            self.model = "text-embedding-004"
            self.dim = 768
        elif backend == "local":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.dim = 384
        else:
            raise ValueError(f"Backend desconocido: {backend}. Usa 'gemini' o 'local'.")
 
        print(f"EmbeddingService iniciado — backend={backend}, dim={self.dim}")
 
    def embed(self, texts: List[str]) -> List[List[float]]:
        """Devuelve una lista de vectores, uno por texto."""
        if self.backend == "gemini":
            result = self.client.models.embed_content(
                model=self.model,
                contents=texts,
            )
            return [e.values for e in result.embeddings]
        else:
            vectors = self.model.encode(texts, normalize_embeddings=True)
            return vectors.tolist()
 
 
embedding_service = EmbeddingService(backend=EMBEDDING_BACKEND)

### 0.2 Elección del backend de generación
| Backend | Modelo | Requiere internet | Costo | Requisito |
|---|---|---|---|---|
| `"gemini"` | `gemini-2.5-flash-lite` | ✅ | Free tier (1500 req/día) | `GOOGLE_API_KEY` |
| `"ollama"` | `qwen2.5:7b` | ❌ | Gratis | Ollama corriendo local |
Cambia `GENERATION_BACKEND` según tu preferencia.

In [3]:
# ── Generación ────────────────────────────────────────────────
GENERATION_BACKEND = "ollama"  # "gemini" | "ollama"


class GenerationService:
    """
    Wrapper unificado para generación de texto.
    Independientemente del backend, el resto del notebook siempre llama:
        service.generate(prompt: str) -> str
    """

    def __init__(self, backend: str = "gemini"):
        self.backend = backend

        if backend == "gemini":
            from google import genai
            self.client = genai.Client(api_key=GOOGLE_API_KEY)
            self.model = "gemini-2.0-flash"
        elif backend == "ollama":
            from langchain_ollama import ChatOllama
            self.client = ChatOllama(model="qwen2.5:7b", temperature=0.1)
        else:
            raise ValueError(f"Backend desconocido: {backend}. Usa 'gemini' o 'ollama'.")

        print(f"GenerationService iniciado — backend={backend}")

    def generate(self, prompt: str, temperature: float = 0.1) -> str:
        """Genera texto a partir de un prompt."""
        if self.backend == "gemini":
            response = self.client.models.generate_content(
                model=self.model,
                contents=prompt,
                config={"temperature": temperature, "max_output_tokens": 512},
            )
            return response.text.strip()
        else:
            from langchain_core.messages import HumanMessage
            self.client.temperature = temperature
            response = self.client.invoke([HumanMessage(content=prompt)])
            return response.content.strip()


generation_service = GenerationService(backend=GENERATION_BACKEND)

# Prueba rápida
test_response = generation_service.generate("Di 'hola' en una palabra.")
print(f"Prueba: {test_response}")

GenerationService iniciado — backend=ollama
Prueba: Hola


### 0.3 Datos del curso
Usaremos documentos internos ficticios de una empresa.
El modelo **no fue entrenado con esta información**, así que no la conoce.

In [4]:
# ── Carga ────────────────────────────────────────────────────

def load_documents(path: str | Path) -> List[Dict]:
    """
    Carga documentos desde un archivo JSON.
    Valida que cada documento tenga los campos mínimos requeridos.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    with path.open(encoding="utf-8") as f:
        documents = json.load(f)

    required_fields = {"id", "title", "content"}
    for i, doc in enumerate(documents):
        missing = required_fields - doc.keys()
        if missing:
            raise ValueError(f"Documento {i} le faltan campos: {missing}")

    return documents

In [5]:
DOCS_PATH = Path("data/company_documents.json")  # ajusta la ruta si es necesario

company_documents = load_documents(DOCS_PATH)

print(f"✅ {len(company_documents)} documentos cargados desde '{DOCS_PATH}'")
print()

for doc in company_documents:
    dept = doc.get("metadata", {}).get("department", "—")
    tipo = doc.get("metadata", {}).get("type", "—")
    print(f"  [{doc['id']}] {doc['title']}")
    print(f"           dept={dept}  type={tipo}  chars={len(doc['content'])}")

✅ 4 documentos cargados desde 'data/company_documents.json'

  [product_001] CloudSync Pro — Plan Enterprise
           dept=product  type=pricing  chars=372
  [policy_001] Política de Trabajo Remoto 2024
           dept=hr  type=policy  chars=350
  [process_001] Proceso de Reembolso a Clientes
           dept=support  type=process  chars=414
  [guide_001] Checklist de Onboarding para Nuevos Empleados
           dept=hr  type=guide  chars=344


In [6]:
# ── Filtrado por metadatos (opcional) ────────────────────────

# Ejemplo: cargar solo documentos de RRHH
hr_docs = [d for d in company_documents if d.get("metadata", {}).get("department") == "hr"]
print(f"\nDocumentos de RRHH: {len(hr_docs)}")
for d in hr_docs:
    print(f"  - {d['title']}")


Documentos de RRHH: 2
  - Política de Trabajo Remoto 2024
  - Checklist de Onboarding para Nuevos Empleados


## Parte 1 — El problema: LLM sin contexto
Antes de construir RAG, veamos qué pasa cuando le hacemos preguntas al modelo
sobre información que no está en su entrenamiento.

In [7]:
test_questions = [
    "¿Cuál es el precio del plan Enterprise de CloudSync Pro?",
    "¿Cuántos días a la semana pueden trabajar remotamente los empleados?",
    "¿Qué pasa con los reembolsos mayores a $100?",
    "¿Qué ocurre durante la primera semana de onboarding?",
]

In [9]:
print("=" * 60)
print("PREGUNTAS AL MODELO SIN CONTEXTO")
print("=" * 60)
 
for i, question in enumerate(test_questions, 1):
    print(f"\nPregunta {i}: {question}")
    response = generation_service.generate(
        f"Responde brevemente: {question}",
        temperature=0.0,
    )
    print(f"Respuesta: {response.strip()}")
    print("-" * 40)

PREGUNTAS AL MODELO SIN CONTEXTO

Pregunta 1: ¿Cuál es el precio del plan Enterprise de CloudSync Pro?
Respuesta: Lo siento, pero no tengo información específica sobre el precio del plan Enterprise de CloudSync Pro. Te recomiendo visitar el sitio web oficial de CloudSync Pro o contactar directamente a su equipo de ventas para obtener la información más precisa y actualizada sobre precios y planes.
----------------------------------------

Pregunta 2: ¿Cuántos días a la semana pueden trabajar remotamente los empleados?
Respuesta: Esa información puede variar dependiendo de las políticas de cada empresa. Sin detalles específicos sobre la organización en cuestión, no se puede determinar un número exacto. Algunas empresas permiten trabajar remoto todos los días, mientras que otras lo permiten solo unos pocos días a la semana.
----------------------------------------

Pregunta 3: ¿Qué pasa con los reembolsos mayores a $100?
Respuesta: Los reembolsos mayores a $100 generalmente requieren ver

**¿Qué observas?**

- El modelo responde con información genérica, inventada o simplemente dice que no sabe.

- Esto es exactamente el problema que RAG resuelve: darle al modelo acceso a **tu información específica** en el momento de la consulta.

## Parte 2 — Internals: embeddings y vector store con FAISS
Antes de usar Qdrant, vamos a construir un vector store desde cero con FAISS.
El objetivo es entender exactamente qué pasa por dentro.
### 2.1 ¿Qué es un embedding?
Un embedding es una función que convierte texto en un vector de números reales.
Textos semánticamente similares quedan cerca en ese espacio vectorial.

In [13]:
# Veamos embeddings en acción
sample_texts = [
    "El gato duerme en el sofá",
    "El felino descansa en el mueble",   # semánticamente similar al anterior
    "La política de reembolsos es clara",  # distinto tema
]
 
vectors = embedding_service.embed(sample_texts)
 
print(f"Dimensión de cada vector: {len(vectors[0])}")
print(f"Primeros 5 valores del vector 1: {[round(v, 4) for v in vectors[0][:5]]}")
 
# Calcular similitud coseno manualmente entre los tres textos
def cosine_sim(a: List[float], b: List[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
 
sim_01 = cosine_sim(vectors[0], vectors[1])
sim_02 = cosine_sim(vectors[0], vectors[2])
 
print(f"\nSimilitud coseno:")
print(f"  texto 0 ↔ texto 1 (similar):  {sim_01:.4f}")
print(f"  texto 0 ↔ texto 2 (distinto): {sim_02:.4f}")

Dimensión de cada vector: 384
Primeros 5 valores del vector 1: [-0.0273, 0.0477, 0.0344, -0.0126, 0.0117]

Similitud coseno:
  texto 0 ↔ texto 1 (similar):  0.5551
  texto 0 ↔ texto 2 (distinto): 0.3550


### 2.2 Vector store manual con FAISS

FAISS es una librería de Facebook para búsqueda eficiente de vectores similares.
Lo usamos aquí para entender los internals; en producción usaremos Qdrant.

In [14]:
import faiss
 
@dataclass
class DocumentChunk:
    """Representa un fragmento de documento con su vector y metadatos."""
    id: str
    content: str
    source_doc_id: str
    title: str
    chunk_index: int
    department: str = ""
    doc_type: str = ""
    embedding: Optional[List[float]] = None
 
 
class FAISSVectorStore:
    """
    Vector store minimalista construido sobre FAISS.
    Propósito: entender los internals de búsqueda vectorial.
    Para producción → ver Parte 4 con Qdrant.
    """
 
    def __init__(self, dim: int):
        # IndexFlatIP = producto interno (equivale a coseno si los vectores están normalizados)
        self.index = faiss.IndexFlatIP(dim)
        self.documents: List[DocumentChunk] = []
        self.id_to_index: Dict[str, int] = {}
        self.dim = dim
 
    def add(self, docs: List[DocumentChunk]) -> None:
        embeddings = []
        for doc in docs:
            if doc.embedding is None:
                raise ValueError(f"Documento {doc.id} no tiene embedding")
 
            # Normalizar → la búsqueda por producto interno equivale a coseno
            vec = np.array(doc.embedding, dtype="float32")
            vec /= np.linalg.norm(vec)
 
            pos = len(self.documents)
            self.documents.append(doc)
            self.id_to_index[doc.id] = pos
            embeddings.append(vec)
 
        self.index.add(np.stack(embeddings))
        print(f"  ➕ {len(docs)} chunks añadidos (total: {self.index.ntotal})")
 
    def search(
        self,
        query_embedding: List[float],
        k: int = 3,
        threshold: float = 0.3,
    ) -> List[Tuple[DocumentChunk, float]]:
        if self.index.ntotal == 0:
            return []
 
        q = np.array(query_embedding, dtype="float32")
        q /= np.linalg.norm(q)
 
        scores, indices = self.index.search(q.reshape(1, -1), k)
 
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0 and score >= threshold:
                results.append((self.documents[idx], float(score)))
        return results

In [15]:
# Prueba rápida
faiss_store = FAISSVectorStore(dim=embedding_service.dim)

In [16]:
# Embeddear y añadir el primer documento como ejemplo
test_chunks_text = [
    "CloudSync Pro Enterprise cuesta $49/mes por usuario con compromiso anual.",
    "El plan mensual sin compromiso está disponible a $59/mes por usuario.",
]
test_vecs = embedding_service.embed(test_chunks_text)

In [17]:
test_chunks = [
    DocumentChunk(
        id=f"test_{i}",
        content=text,
        source_doc_id="product_001",
        title="CloudSync Pro",
        chunk_index=i,
        embedding=vec,
    )
    for i, (text, vec) in enumerate(zip(test_chunks_text, test_vecs))
]
faiss_store.add(test_chunks)

  ➕ 2 chunks añadidos (total: 2)


In [18]:
# Búsqueda de prueba
query_vec = embedding_service.embed(["¿Cuánto cuesta el plan anual?"])[0]
results = faiss_store.search(query_vec, k=2, threshold=0.2)
 
print("\nResultados de búsqueda en FAISS:")
for doc, score in results:
    print(f"  [{score:.4f}] {doc.content}")


Resultados de búsqueda en FAISS:
  [0.6036] El plan mensual sin compromiso está disponible a $59/mes por usuario.
  [0.3657] CloudSync Pro Enterprise cuesta $49/mes por usuario con compromiso anual.


## Parte 3 — Estrategias de chunking

No podemos meter documentos completos en cada query: hay límites de contexto y el ruido baja la calidad de las respuestas. Necesitamos dividir los documentos en fragmentos (*chunks*) antes de indexarlos.

Vamos a comparar dos estrategias:
 

In [19]:
def simple_chunker(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Chunking por caracteres con overlap.
    Ventaja: tamaño predecible.
    Desventaja: puede cortar a mitad de oración.
    """
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # Intentar terminar en punto si es posible
        if end < len(text):
            last_period = text.rfind(".", start, end)
            if last_period > start + chunk_size // 2:
                end = last_period + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        next_start = end - overlap
        start = max(next_start, start + 1)
        if end >= len(text):
            break
    return chunks

In [20]:
def semantic_chunker(text: str, max_chunk_size: int = 400) -> List[str]:
    """
    Chunking semántico: respeta párrafos y oraciones.
    Ventaja: preserva unidades de significado.
    Desventaja: tamaño variable.
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    # Si no hay doble salto de línea, dividir por oraciones
    if len(paragraphs) == 1:
        paragraphs = [s.strip() + "." for s in text.split(".") if s.strip()]
 
    chunks, current = [], ""
    for para in paragraphs:
        if current and len(current) + len(para) + 2 > max_chunk_size:
            chunks.append(current.strip())
            current = para
        else:
            current = (current + "\n\n" + para).strip() if current else para
 
    if current.strip():
        chunks.append(current.strip())
    return chunks

In [21]:
# ── Comparación sobre el documento de pricing ──
sample = company_documents[0]
print(f"Documento: '{sample['title']}'")
print(f"Longitud original: {len(sample['content'])} caracteres\n")
 
simple_chunks = simple_chunker(sample["content"], chunk_size=200, overlap=40)
print(f"1. Chunking simple → {len(simple_chunks)} chunks:")
for i, c in enumerate(simple_chunks):
    print(f"   Chunk {i+1} ({len(c)} chars): {c[:80]}...")
 
semantic_chunks = semantic_chunker(sample["content"], max_chunk_size=250)
print(f"\n2. Chunking semántico → {len(semantic_chunks)} chunks:")
for i, c in enumerate(semantic_chunks):
    print(f"   Chunk {i+1} ({len(c)} chars): {c[:80]}...")
 
print("\n📌 Trade-offs:")
print("  Simple:   tamaño predecible, puede romper oraciones")
print("  Semántico: preserva significado, tamaño variable")

Documento: 'CloudSync Pro — Plan Enterprise'
Longitud original: 372 caracteres

1. Chunking simple → 3 chunks:
   Chunk 1 (187 chars): CloudSync Pro Enterprise ofrece almacenamiento ilimitado, colaboración en tiempo...
   Chunk 2 (155 chars): 49/mes por usuario con compromiso anual. Incluye: control de versiones avanzado,...
   Chunk 3 (110 chars): n SSO, y SLA de 99.9% de disponibilidad. El plan mensual sin compromiso está dis...

2. Chunking semántico → 2 chunks:
   Chunk 1 (188 chars): CloudSync Pro Enterprise ofrece almacenamiento ilimitado, colaboración en tiempo...
   Chunk 2 (187 chars): Incluye: control de versiones avanzado, registros de auditoría, integración SSO,...

📌 Trade-offs:
  Simple:   tamaño predecible, puede romper oraciones
  Semántico: preserva significado, tamaño variable


## Parte 4 — RAG completo con Qdrant

Ahora construimos el pipeline RAG de producción usando Qdrant como vector store.
Qdrant ofrece filtrado por metadatos, persistencia y una API limpia.

Usamos el modo **in-memory** para el lab (sin servidor externo).

In [22]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
 
COLLECTION_NAME = "company_docs"

In [23]:
# Cliente in-memory — en producción: QdrantClient(url="http://localhost:6333")
qdrant = QdrantClient(":memory:")
 
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embedding_service.dim,
        distance=Distance.COSINE,
    ),
)
print(f"Colección '{COLLECTION_NAME}' creada (dim={embedding_service.dim})")

Colección 'company_docs' creada (dim=384)


In [24]:
def ingest_documents(
    documents: List[Dict],
    qdrant_client: QdrantClient,
    emb_service: EmbeddingService,
    collection: str = COLLECTION_NAME,
    max_chunk_size: int = 350,
) -> int:
    """
    Pipeline de ingestión:
      1. Chunkear cada documento con chunking semántico
      2. Embeddear todos los chunks en batch
      3. Insertar en Qdrant con metadatos
 
    Retorna el número total de chunks insertados.
    """
    all_texts, all_meta = [], []
 
    for doc in documents:
        chunks = semantic_chunker(doc["content"], max_chunk_size=max_chunk_size)
        for i, chunk_text in enumerate(chunks):
            if len(chunk_text) < 20:   # descartar chunks vacíos
                continue
            all_texts.append(chunk_text)
            all_meta.append({
                "chunk_id": f"{doc['id']}_chunk_{i}",
                "source_doc_id": doc["id"],
                "title": doc["title"],
                "chunk_index": i,
                "department": doc["metadata"].get("department", ""),
                "doc_type": doc["metadata"].get("type", ""),
                "content": chunk_text,   # guardamos el texto para recuperarlo
            })
 
    # Embeddear en batch (más eficiente que uno por uno)
    print(f"  Generando embeddings para {len(all_texts)} chunks...")
    vectors = emb_service.embed(all_texts)
 
    # Insertar en Qdrant
    points = [
        PointStruct(id=idx, vector=vec, payload=meta)
        for idx, (vec, meta) in enumerate(zip(vectors, all_meta))
    ]
    qdrant_client.upsert(collection_name=collection, points=points)
 
    print(f"  ✅ {len(points)} chunks indexados en Qdrant")
    return len(points)

In [25]:
total = ingest_documents(company_documents, qdrant, embedding_service)
print(f"\nTotal chunks en el índice: {total}")

  Generando embeddings para 7 chunks...
  ✅ 7 chunks indexados en Qdrant

Total chunks en el índice: 7


In [26]:
def retrieve(
    question: str,
    qdrant_client: QdrantClient,
    emb_service: EmbeddingService,
    k: int = 3,
    score_threshold: float = 0.3,
    collection: str = COLLECTION_NAME,
) -> List[Dict]:
    """
    Recupera los k chunks más relevantes para la pregunta.
    Retorna lista de payloads (con 'content', 'title', 'score').
    """
    query_vec = emb_service.embed([question])[0]
 
    hits = qdrant_client.query_points(
        collection_name=collection,
        query=query_vec,
        limit=k,
        score_threshold=score_threshold,
    ).points
 
    results = []
    for hit in hits:
        payload = hit.payload.copy()
        payload["score"] = hit.score
        results.append(payload)
    return results

In [29]:
def ask(
    question: str,
    qdrant_client: QdrantClient,
    emb_service: EmbeddingService,
    temperature: float = 0.1,
    k: int = 3,
    score_threshold: float = 0.3,
    system_role: str = "asistente empresarial",
) -> Tuple[str, List[Dict]]:
    """
    Pipeline RAG completo:
      1. Retrieve — buscar chunks relevantes
      2. Augment  — construir prompt con contexto
      3. Generate — llamar a GenerationService
 
    Retorna (respuesta, chunks_usados) para poder evaluar.
    """
    # 1. RETRIEVE
    chunks = retrieve(question, qdrant_client, emb_service, k=k, score_threshold=score_threshold)
    if not chunks:
        return "No encontré información relevante en los documentos.", []
 
    # 2. AUGMENT
    context = "\n\n---\n\n".join(
        f"[{c['title']}]\n{c['content']}" for c in chunks
    )
    prompt = f"""Eres un {system_role}. Responde la pregunta basándote ÚNICAMENTE en el contexto proporcionado.
Si la información no está en el contexto, di explícitamente que no la tienes.
 
CONTEXTO:
{context}
 
PREGUNTA: {question}
 
RESPUESTA:"""
 
    # 3. GENERATE
    answer = generation_service.generate(prompt, temperature=temperature)
    return answer.strip(), chunks

In [30]:
# ── Prueba del pipeline completo ──
print("=" * 60)
print("RAG CON QDRANT — PIPELINE COMPLETO")
print("=" * 60)
 
for question in test_questions:
    answer, used_chunks = ask(question, qdrant, embedding_service)
    sources = list({c["title"] for c in used_chunks})
    print(f"\n❓ {question}")
    print(f"💬 {answer}")
    print(f"📎 Fuentes: {sources}")
    print("-" * 40)

RAG CON QDRANT — PIPELINE COMPLETO



❓ ¿Cuál es el precio del plan Enterprise de CloudSync Pro?
💬 El precio del plan Enterprise de CloudSync Pro es de $49/mes por usuario con compromiso anual. También se ofrece un plan mensual sin compromiso a $59/mes por usuario.
📎 Fuentes: ['Checklist de Onboarding para Nuevos Empleados', 'CloudSync Pro — Plan Enterprise']
----------------------------------------

❓ ¿Cuántos días a la semana pueden trabajar remotamente los empleados?
💬 Los empleados pueden trabajar de forma remota hasta 3 días por semana.
📎 Fuentes: ['Proceso de Reembolso a Clientes', 'Política de Trabajo Remoto 2024', 'Checklist de Onboarding para Nuevos Empleados']
----------------------------------------

❓ ¿Qué pasa con los reembolsos mayores a $100?
💬 Para reembolsos mayores a $100, según el proceso proporcionado, se requiere aprobación del supervisor.
📎 Fuentes: ['Proceso de Reembolso a Clientes', 'Política de Trabajo Remoto 2024', 'CloudSync Pro — Plan Enterprise']
----------------------------------------

❓ ¿Qu

## Parte 5 — Tuning: prompts y similarity threshold

Dos palancas principales para mejorar la calidad del RAG:
1. **El prompt** — el rol y las instrucciones que le damos al modelo
2. **El threshold** — cuán estrictos somos al filtrar chunks por relevancia

In [31]:
# ── 5.1 Efecto del rol en el prompt ──
question = "¿Cuál es la política de trabajo remoto?"
 
print("MISMO CONTEXTO, DIFERENTE ROL EN EL PROMPT\n")
 
roles = {
    "asistente de RRHH formal": 0.1,
    "agente de atención al cliente amigable": 0.4,
}
 
for role, temp in roles.items():
    answer, _ = ask(question, qdrant, embedding_service, temperature=temp, system_role=role)
    print(f"🎭 Rol: {role} (temp={temp})")
    print(f"   {answer}\n")

MISMO CONTEXTO, DIFERENTE ROL EN EL PROMPT



🎭 Rol: asistente de RRHH formal (temp=0.1)
   Según el contexto proporcionado, la política de trabajo remoto 2024 establece lo siguiente:

- Los empleados pueden trabajar de forma remota hasta 3 días por semana.
- El trabajo remoto requiere aprobación del jefe directo.

No se incluyen otras aspectos relacionados con el trabajo remoto en el contexto dado.

🎭 Rol: agente de atención al cliente amigable (temp=0.4)
   Según la información proporcionada, la política de trabajo remoto para 2024 permite a los empleados trabajar de forma remota hasta 3 días por semana. Esta política requiere aprobación del jefe directo y se aplica a partir de enero de 2024.



In [32]:
# ── 5.2 Efecto del similarity threshold ──
query_test = "trabajo remoto empleados"
query_vec = embedding_service.embed([query_test])[0]
 
print(f"EFECTO DEL THRESHOLD — query: '{query_test}'\n")
print(f"{'Threshold':>10} | {'Resultados':>10} | Scores")
print("-" * 50)
 
for threshold in [0.1, 0.3, 0.5, 0.7]:
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        limit=5,
        score_threshold=threshold,
    ).points
    scores = [round(h.score, 3) for h in hits]
    print(f"{threshold:>10} | {len(hits):>10} | {scores}")
 
print("""
Regla práctica:
  < 0.3  → demasiado ruido, contexto irrelevante
  0.3–0.5 → buen balance para documentos empresariales
  > 0.5  → puede perder resultados relevantes con distinto vocabulario
""")

EFECTO DEL THRESHOLD — query: 'trabajo remoto empleados'

 Threshold | Resultados | Scores
--------------------------------------------------
       0.1 |          5 | [0.505, 0.487, 0.44, 0.364, 0.358]
       0.3 |          5 | [0.505, 0.487, 0.44, 0.364, 0.358]
       0.5 |          1 | [0.505]
       0.7 |          0 | []

Regla práctica:
  < 0.3  → demasiado ruido, contexto irrelevante
  0.3–0.5 → buen balance para documentos empresariales
  > 0.5  → puede perder resultados relevantes con distinto vocabulario




## Parte 6 — Evaluación con DeepEval

Hasta ahora hemos evaluado el RAG "a ojo". DeepEval nos da métricas automáticas:

| Métrica | ¿Qué mide? |
|---|---|
| `faithfulness` | ¿La respuesta está soportada por el contexto recuperado? |
| `answer_relevancy` | ¿La respuesta es relevante para la pregunta? |

Ambas métricas usan un LLM como juez internamente.

In [62]:
from deepeval.models import GeminiModel
from deepeval.models import OllamaModel
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams
from deepeval import evaluate

In [45]:
# ── Modelo juez ───────────────────────────────────────────────
# DeepEval usa Gemini directamente via google-genai — sin dependencias extra
load_dotenv(override=True)
GOOGLE_API_KEY_EVAL = os.getenv("GEMINI_API_KEY_EVAL")
eval_model = GeminiModel(
    model="gemini-2.0-flash-lite",
    api_key=GOOGLE_API_KEY_EVAL,
)
 
print("Modelo juez listo:", eval_model.get_model_name())

Modelo juez listo: gemini-2.0-flash-lite (Gemini)


In [49]:
eval_model = OllamaModel(
    model="qwen2.5:7b",
    base_url="http://localhost:11434",
    temperature=0,
)
print(f"✅ Modelo juez: {eval_model.get_model_name()}")

✅ Modelo juez: qwen2.5:7b (Ollama)


### 6.1 Sin ground truth

`FaithfulnessMetric` y `AnswerRelevancyMetric` solo necesitanla pregunta, la respuesta y los chunks recuperados.

In [50]:
# ── Métricas sin ground truth ─────────────────────────────────
faithfulness_metric    = FaithfulnessMetric(model=eval_model, threshold=0.7)
answer_relevancy_metric = AnswerRelevancyMetric(model=eval_model, threshold=0.7)

In [51]:
# ── Construir test cases ──────────────────────────────────────
print("Generando respuestas RAG...")
test_cases_no_gt = []
for question in test_questions:
    answer, chunks = ask(question, qdrant, embedding_service)
    test_cases_no_gt.append(LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[c["content"] for c in chunks],
        # sin expected_output
    ))
    print(f"  ✅ '{question[:55]}...'")

Generando respuestas RAG...


  ✅ '¿Cuál es el precio del plan Enterprise de CloudSync Pro...'
  ✅ '¿Cuántos días a la semana pueden trabajar remotamente l...'
  ✅ '¿Qué pasa con los reembolsos mayores a $100?...'
  ✅ '¿Qué ocurre durante la primera semana de onboarding?...'


In [52]:
print("\nEjecutando evaluación...")
results_no_gt = evaluate(
    test_cases=test_cases_no_gt,
    metrics=[faithfulness_metric, answer_relevancy_metric],
)


Ejecutando evaluación...


✨ You're running DeepEval's latest Faithfulness Metric! (using qwen2.5:7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen2.5:7b (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ❌ Faithfulness (score: 0.0, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b (Ollama), reason: The score is 0.00 because the actual output incorrectly includes information about an onboarding process and training modules, which are not present in the retrieval context., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b (Ollama), reason: The score is 1.00 because there are no irrelevant statements in the actual output, making it a perfect score., error: None)

For test case:

  - input: ¿Qué ocurre durante la primera semana de onboarding?
  - actual output: Durante la primera semana del onboarding, los nuevos empleados deben completar módulos de entrenamiento obligatorios que cubren temas como seguridad, cumplimiento y cultura de la empresa.
  - expected output: None
  - context: None
  - retrieval context: ['Día 1: Configuración de TI y acceso a sistemas.\n\nDía 2: Orientación departamental con

⚠ WARNING: No hyperparameters logged.
» ]8;id=879820;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 113.25s | token cost: None)
» Test Results (4 total tests):
   » Pass Rate: 50.0% | Passed: 2 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [53]:
ground_truths = [
    # ¿Cuál es el precio del plan Enterprise de CloudSync Pro?
    "El plan Enterprise de CloudSync Pro cuesta $49/mes por usuario con compromiso anual. "
    "Sin compromiso, el plan mensual está disponible a $59/mes por usuario.",

    # ¿Cuántos días a la semana pueden trabajar remotamente los empleados?
    "Los empleados pueden trabajar de forma remota hasta 3 días por semana, "
    "con aprobación del jefe directo.",

    # ¿Qué pasa con los reembolsos mayores a $100?
    "Los reembolsos mayores a $100 requieren aprobación del supervisor antes de procesarse. "
    "Una vez aprobados, se procesan en 3–5 días hábiles al método de pago original.",

    # ¿Qué ocurre durante la primera semana de onboarding?
    "Durante la primera semana se completan los módulos de entrenamiento obligatorios "
    "de seguridad, cumplimiento y cultura de la empresa.",
]

In [63]:
faithfulness_metric     = FaithfulnessMetric(model=eval_model, threshold=0.7)
answer_relevancy_metric = AnswerRelevancyMetric(model=eval_model, threshold=0.7)
correctness_metric      = GEval(
    name="Correctness",
    criteria="Determina si el actual_output es factualmente correcto comparado con el expected_output.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=eval_model,
    threshold=0.7,
)


In [64]:
# Test cases con ground truth
test_cases = []
for question, gt in zip(test_questions, ground_truths):
    answer, chunks = ask(question, qdrant, embedding_service)
    test_cases.append(LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[c["content"] for c in chunks],
        expected_output=gt,
    ))

In [66]:
results_gt = evaluate(
    test_cases=test_cases,
    metrics=[faithfulness_metric, answer_relevancy_metric, correctness_metric],
)

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen2.5:7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen2.5:7b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using qwen2.5:7b (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b (Ollama), reason: The score is 1.00 because there are no contradictions in the actual output, indicating it perfectly aligns with the retrieval context., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b (Ollama), reason: The score is 1.00 because there are no irrelevant statements in the actual output, making it eligible for a perfect score., error: None)
  - ✅ Correctness [GEval] (score: 0.8, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b (Ollama), reason: The response accurately matches the factual content of the expected output, but fails to include the additional detail about approval from the direct supervisor. This omission results in a score below perfect due to incomplete alignment with all aspects of the expected output., error: None)

For test case:

  - input: ¿Cuántos días a la semana pueden t

⚠ WARNING: No hyperparameters logged.
» ]8;id=523002;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 121.69s | token cost: None)
» Test Results (4 total tests):
   » Pass Rate: 25.0% | Passed: 1 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.